In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [17]:
from pyspark.sql.functions import *

# Students Who Failed All Exams

## Problem Description

You are given two DataFrames:

- **`students`**:
  - `student_id` (int): Unique student identifier
  - `student_name` (str): Name of the student

- **`exams`**:
  - `exam_id` (int): Exam identifier
  - `student_id` (int): Student who took the exam
  - `score` (int): Score achieved (0‑100)

**Task:**  
A student fails an exam if their score is **strictly below 60**.  
Find students who failed **all** of their exams – i.e., every single exam score they have is below 60.

**Output:**  
Return a DataFrame with columns `student_id` and `student_name`.

---

## Example

**Input Data**

`students`:

| student_id | student_name |
|------------|--------------|
| 1          | Alice        |
| 2          | Bob          |
| 3          | Charlie      |
| 4          | Diana        |

`exams`:

| exam_id | student_id | score |
|---------|------------|-------|
| 1       | 1          | 45    |
| 2       | 1          | 30    |
| 3       | 1          | 55    |
| 4       | 2          | 70    |
| 5       | 2          | 40    |
| 6       | 3          | 20    |
| 7       | 3          | 15    |
| 8       | 4          | 80    |
| 9       | 4          | 90    |

**Expected Output:**

| student_id | student_name |
|------------|--------------|
| 1          | Alice        |
| 3          | Charlie      |

- Alice: all scores (45, 30, 55) < 60 → qualifies.  
- Bob: has a score of 70 (≥ 60) → does not qualify.  
- Charlie: all scores (20, 15) < 60 → qualifies.  
- Diana: all scores (80, 90) ≥ 60 → does not qualify.


## Dataset Generation

In [4]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define schema for students
students_schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True)
])

students_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "Diana")
]

students_df = spark.createDataFrame(students_data, schema=students_schema)

# Define schema for exams
exams_schema = StructType([
    StructField("exam_id", IntegerType(), True),
    StructField("student_id", IntegerType(), True),
    StructField("score", IntegerType(), True)
])

exams_data = [
    (1, 1, 45),
    (2, 1, 30),
    (3, 1, 55),
    (4, 2, 70),
    (5, 2, 40),
    (6, 3, 20),
    (7, 3, 15),
    (8, 4, 80),
    (9, 4, 90)
]

exams_df = spark.createDataFrame(exams_data, schema=exams_schema)

students_df.show()
exams_df.show()

+----------+------------+
|student_id|student_name|
+----------+------------+
|         1|       Alice|
|         2|         Bob|
|         3|     Charlie|
|         4|       Diana|
+----------+------------+

+-------+----------+-----+
|exam_id|student_id|score|
+-------+----------+-----+
|      1|         1|   45|
|      2|         1|   30|
|      3|         1|   55|
|      4|         2|   70|
|      5|         2|   40|
|      6|         3|   20|
|      7|         3|   15|
|      8|         4|   80|
|      9|         4|   90|
+-------+----------+-----+



In [5]:
students_df.createOrReplaceTempView("students")
exams_df.createOrReplaceTempView("exams")

In [14]:
spark.sql(
    """
      with cte as (
            SELECT student_id, max(score) as max_score 
            from exams 
            group by student_id
            )
       SELECT s.student_id,s.student_name 
       from students s 
       join cte c on s.student_id = c.student_id
       WHERE c.max_score < 60
       
    """
).show()

+----------+------------+
|student_id|student_name|
+----------+------------+
|         1|       Alice|
|         3|     Charlie|
+----------+------------+



# Using pyspark

In [23]:
exams_df.groupBy("student_id") \
        .agg(max(col("score")).alias("max_score")) \
        .filter(col("max_score") < 60) \
        .join(students_df, on="student_id") \
        .select("student_id", "student_name") \
        .show()

+----------+------------+
|student_id|student_name|
+----------+------------+
|         1|       Alice|
|         3|     Charlie|
+----------+------------+

